In [ ]:
# ======================================================================
# MODULO A — Setup & Parametri
# A.1 — Definizione dei parametri globali del modello e della pipeline
# ======================================================================


# === COLAB ===
from google.colab import files
import os

OUTPUT_DIR = "/content/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VERBOSE = False  # se True mostra tutti i log

ID_COL        = "SKU"                    # chiave prodotto
DESC_COL      = "Description"            # descrizione
LT_COL_NAME   = "LT"                     # lead time
PACK_SIZE_COL = "Round"                  # multiplo d’imballo
UOM_COL       = "BUn"                    # unità di misura

# === 🔮 FORECAST FUTURO ===
HORIZON = 25                             # mesi da prevedere
TRIM_LEADING_ZEROS = True                # ignora zeri iniziali
MIN_HISTORY_POINTS = 6                   # min punti richiesti

# === 🔙 BACKTEST ===
HORIZON_BACKTEST = 12                    # mesi di backtest (ultimo tratto da simulare)
# Griglia di quantili che useremo in seguito per ottimizzare (per ora solo definita)
QUANTILE_GRID = [0.10, 0.20, 0.30, 0.40,
                 0.50, 0.60, 0.70, 0.80, 0.90]

# === 🧹 OUTLIERS (winsorizing su serie storica) ===
REMOVE_OUTLIERS = True                   # abilita/disabilita filtro
OUTLIER_LEVEL   = 0.05                   # 5% low / 95% high

# === 🎯 ARROTONDAMENTI ===
ROUNDING_MODE    = "nearest"             # "up" | "down" | "nearest"
ROUND_DECIMALS   = 3                     # decimali dopo arrotondamento

# === 🎛️ CALIBRAZIONE (opzionale, come logica esistente) ===
# Mesi su cui applicare l’aggiustamento (es. agosto, dicembre)
CALIBRATION_MONTHS = [8, 12]             # [] = disattivata

# === 📦 INVENTORY & SAFETY STOCK ===
CALCULATE_SS       = True                # Attiva/Disattiva calcolo
DEFAULT_LEAD_TIME  = 30                  # Giorni (se colonna LT manca)
REORDER_PERIOD     = 30                  # Giorni (1 mese fisso come da richiesta)
SS_LOOKBACK_MONTHS = 12                  # Mesi storici per deviazione standard

# Soglie ABC (Cumulativa Volumi)
ABC_LIMITS = {
    "A": 0.70,  # Top 70% volume
    "B": 0.90,  # 70% -> 90%
    "C": 1.00   # Resto
}

# Soglie XYZ (Coefficiente Variazione CV = StdDev / Mean)
XYZ_LIMITS = {
    "X": 0.40,  # CV <= 0.4 (Molto stabile)
    "Y": 0.80,  # 0.4 < CV <= 0.8 (Variabile)
    "Z": 999.0  # CV > 0.8 (Erratico)
}

# Livelli di Servizio Target (Target Cycle Service Level)
# Mappatura Classe -> % Service Level
SERVICE_LEVEL_MATRIX = {
    "AX": 0.97, "AY": 0.95, "AZ": 0.93,
    "BX": 0.91, "BY": 0.90, "BZ": 0.89,
    "CX": 0.87, "CY": 0.80, "CZ": 0.00   # 0.00 = No Safety Stock
}

# === 📤 OUTPUT ===
OUTPUT_FORMAT     = "xlsx"
OUTPUT_FILE_BASE  = "Forecast and SS"
OUTPUT_SUFFIX     = " v0"                # suffisso nome file


In [ ]:
# ======================================================================
# MODULO A — Setup & Parametri
# A.2 — Import delle librerie principali
# ======================================================================

import pandas as pd
import numpy as np
import re
import math


In [ ]:
# ============================================================
# MODULO B — CARICAMENTO & PREPROCESSING (COLAB VERSION)
# B.1 — Caricamento file Excel da upload
# ============================================================

from google.colab import files
import pandas as pd

print("📥 Carica il file Excel dal tuo PC...")
uploaded = files.upload()  # ti apre la finestra per scegliere il file

# Nome del file caricato
INPUT_FILE = list(uploaded.keys())[0]

# Leggo il primo foglio disponibile
all_sheets = pd.read_excel(INPUT_FILE, sheet_name=None)
first_sheet_name = list(all_sheets.keys())[0]
df_raw = all_sheets[first_sheet_name]

print("✔️ Foglio caricato:", first_sheet_name)
df_raw.head()


In [ ]:
# ======================================================================
# MODULO B — Caricamento & Preprocessing
# B.2 — Identificazione automatica delle colonne temporali (YYYY_MM)
# ======================================================================

date_col_pattern = re.compile(r"^\d{4}_\d{2}$")

date_cols = [col for col in df_raw.columns if date_col_pattern.match(str(col))]
meta_cols = [col for col in df_raw.columns if col not in date_cols]

print("Colonne temporali trovate:", len(date_cols))
print(date_cols[:10], "...")
print("\nColonne meta:", meta_cols)


In [ ]:
# ======================================================================
# MODULO B — Caricamento & Preprocessing
# B.3 — FIX: conversione NaN → 0 nelle colonne temporali
# ======================================================================

print("➡️ FIX: converto i NaN delle colonne temporali in 0.")
df_raw[date_cols] = df_raw[date_cols].fillna(0)
print("✔️ NaN convertiti in 0.")


In [ ]:
# ======================================================================
# MODULO B — Caricamento & Preprocessing
# B.4 — Conversione WIDE → LONG
# ======================================================================

df_long = df_raw.melt(
    id_vars=meta_cols,
    value_vars=date_cols,
    var_name="Period",
    value_name="Demand"
)

df_long = df_long.dropna(subset=["Demand"]).reset_index(drop=True)

print(df_long.head())
print("\nShape finale:", df_long.shape)


In [ ]:
# ============================================================
# MODULO B — CARICAMENTO & PREPROCESSING
# B.5 — Parsing periodo → datetime e ordinamento
# ============================================================

df_long["Date"] = pd.to_datetime(df_long["Period"], format="%Y_%m")
df_long = df_long.sort_values(by=[ID_COL, "Date"]).reset_index(drop=True)

print(df_long[[ID_COL, "Period", "Date", "Demand"]].head(10))


In [ ]:
# ============================================================
# MODULO B — CARICAMENTO & PREPROCESSING
# B.6 — Filtro SKU con storico insufficiente
# ============================================================

hist_counts = (
    df_long.groupby(ID_COL)["Demand"]
    .count()
    .reset_index(name="history_points")
)

valid_skus = hist_counts[hist_counts["history_points"] >= MIN_HISTORY_POINTS][ID_COL]

print(f"SKU totali: {df_long[ID_COL].nunique()}")
print(f"SKU con storico >= {MIN_HISTORY_POINTS}: {len(valid_skus)}")

df_filtered = (
    df_long[df_long[ID_COL].isin(valid_skus)]
    .reset_index(drop=True)
    .copy()
)

print("\nShape finale dati filtrati:", df_filtered.shape)
df_filtered.head(10)


In [ ]:
# ============================================================
# MODULO B — CARICAMENTO & PREPROCESSING
# B.7 — Winsorizing outlier per SKU
# ============================================================

def winsorize_series(s, level=OUTLIER_LEVEL):
    """
    Applica winsorizing a una serie: taglia valori sotto il quantile `level`
    e sopra il quantile `1-level`. Se REMOVE_OUTLIERS è False, ritorna s.
    """
    if not REMOVE_OUTLIERS:
        return s

    vals = s.dropna()
    if len(vals) == 0:
        return s

    q_low = vals.quantile(level)
    q_high = vals.quantile(1 - level)

    return s.clip(lower=q_low, upper=q_high)

print("➡️ Applicazione winsorizing per SKU...")
df_filtered["Demand"] = (
    df_filtered
    .groupby(ID_COL)["Demand"]
    .transform(lambda s: winsorize_series(s, OUTLIER_LEVEL))
)
print("✔️ Winsorizing completato.")
df_filtered.head()


In [ ]:
# ============================================================
# MODULO C — SERIE STORICHE
# C.1 — Costruzione delle serie storiche complete (formato TimesFM)
# ============================================================

df_mode = df_filtered.copy()

print("➡️ Utilizzo dell'intero storico disponibile per il forecasting.")
print("SKU totali:", df_mode[ID_COL].nunique())
print("Righe totali:", len(df_mode))

sku_series = {}   # SKU → lista valori storici

for sku, group in df_mode.groupby(ID_COL):
    g = group.sort_values("Date")
    values = g["Demand"].astype(float).tolist()
    dates  = g["Date"].tolist()

    # opzionale: rimozione zeri iniziali
    if TRIM_LEADING_ZEROS:
        idx = 0
        while idx < len(values) and values[idx] == 0:
            idx += 1
        values = values[idx:]
        dates  = dates[idx:]

    if len(values) == 0:
        continue

    sku_series[sku] = values

print("Serie generate per", len(sku_series), "SKU.")
first_sku = list(sku_series.keys())[0]
print("\nEsempio prima serie:")
print("SKU:", first_sku)
print("Serie:", sku_series[first_sku][:10], "...")
print("Lunghezza:", len(sku_series[first_sku]))


In [ ]:
# ============================================================
# MODULO C — SERIE STORICHE
# C.2 — Preparazione serie storiche troncate per il backtest
# ============================================================

backtest_series = {}   # SKU -> lista valori storici TRONCATI (input a TimesFM nel backtest)
backtest_actuals = {}  # SKU -> np.array ultimi HORIZON_BACKTEST valori reali (target backtest)

skus_with_series = len(sku_series)
skipped_too_short = 0

print(f"➡️ Preparazione serie per backtest con HORIZON_BACKTEST = {HORIZON_BACKTEST} mesi")

for sku, values in sku_series.items():
    n = len(values)

    # Serve almeno HORIZON_BACKTEST per avere dati di confronto
    if n <= HORIZON_BACKTEST:
        skipped_too_short += 1
        continue

    # Storico troncato (da dare a TimesFM per simulare il backtest)
    hist_bt = values[:-HORIZON_BACKTEST]
    # Valori reali degli ultimi HORIZON_BACKTEST mesi (per il calcolo dell'accuratezza)
    act_bt = values[-HORIZON_BACKTEST:]

    # Facciamo anche qui un check minimo sulla lunghezza dello storico troncato
    if len(hist_bt) < MIN_HISTORY_POINTS:
        skipped_too_short += 1
        continue

    backtest_series[sku] = hist_bt
    backtest_actuals[sku] = np.array(act_bt, dtype=float)

print("\n📊 Riepilogo preparazione backtest:")
print(f" - SKU totali con serie storica (post-trim): {skus_with_series}")
print(f" - SKU utilizzabili per backtest:           {len(backtest_series)}")
print(f" - SKU esclusi (serie troppo corte):        {skipped_too_short}")


In [ ]:
# ============================================================
# MODULO C — SERIE STORICHE
# C.3 — Funzioni di accuratezza Motul (mensile e pesata)
# ============================================================

def accuracy_single_month(act, fcst):
    """
    Accuratezza mensile secondo la formula ufficiale:

        Δ = |ACT - FCST|
        ACC_i = 0 se:
            - ACT <= 0
            - FCST <= 0
            - Δ > ACT
            - Δ > FCST
        altrimenti:
            1 - Δ / ACT
    """
    act = float(act)
    fcst = float(fcst)

    # Condizioni che annullano l'accuranza
    if act <= 0:
        return 0.0
    if fcst <= 0:
        return 0.0

    delta = abs(act - fcst)

    if delta > act:
        return 0.0
    if delta > fcst:
        return 0.0

    return 1 - (delta / act)


def accuracy_weighted(act_array, fcst_array):
    """
    ACCURANCY PESATA secondo la formula:

        ACC = sum( ACC_i * (ACT_i + FCST_i) ) / sum(ACT_i + FCST_i)

    Restituisce 0 se i pesi totali sono nulli.
    """
    acc_list = []
    weights = []

    for act, fc in zip(act_array, fcst_array):
        acc_i = accuracy_single_month(act, fc)
        w_i = act + fc
        acc_list.append(acc_i * w_i)
        weights.append(w_i)

    total_w = sum(weights)
    if total_w <= 0:
        return 0.0

    return sum(acc_list) / total_w


In [ ]:
# ============================================================
# MODULO D — CALIBRAZIONE
# D.1 — Calcolo fattori di calibrazione (globali + per SKU)
# ============================================================

# Se non ci sono mesi da calibrare, mappa vuota
if not CALIBRATION_MONTHS:
    print("ℹ️ CALIBRATION_MONTHS vuoto: nessuna calibrazione calcolata.")
    CALIBRATION_FACTORS = {}
    GLOBAL_CALIBRATION_FACTORS = {}
else:
    def month_from_label(label):
        try:
            return int(str(label).split("_")[1])
        except Exception:
            return None

    def theil_sen_log_trend(values):
        """
        Calcola pendenza (beta) e intercetta (alpha) robusta su scala logaritmica.
        Usa tutte le coppie di punti (Theil-Sen completo).
        Accetta serie con zeri interni: li filtra internamente preservando
        le posizioni temporali originali, così i residui risultano coerenti.
        """
        y = np.array(values, dtype=float)
        valid_idx = np.where(y > 0)[0]
        if len(valid_idx) < 6:
            return None, None

        y_log = np.log(y[valid_idx])
        t = valid_idx.astype(float)

        n = len(y_log)
        slopes = []
        for i in range(n):
            for j in range(i + 1, n):
                denom = t[j] - t[i]
                if denom != 0:
                    slopes.append((y_log[j] - y_log[i]) / denom)

        if not slopes:
            return None, None

        beta = float(np.median(slopes))
        alpha = float(np.median(y_log - beta * t))
        return alpha, beta

    # Storico wide
    df_hist_wide = df_filtered.pivot(
        index=ID_COL,
        columns="Period",
        values="Demand"
    ).reset_index()

    time_cols = [c for c in df_hist_wide.columns if re.match(r"^\d{4}_\d{2}$", str(c))]
    time_cols = sorted(time_cols)

    # --- fallback globale: mediana residui log per mese target ---
    def global_month_medians(df_wide, time_cols, target_months):
        med = {m: [] for m in target_months}

        for _, row in df_wide.iterrows():
            vals = pd.Series([row[c] for c in time_cols], index=time_cols, dtype="float")
            vals = pd.to_numeric(vals, errors="coerce")

            # trim solo zeri iniziali, mantieni zeri interni
            nz = np.where((vals.fillna(0) > 0).values)[0]
            if len(nz) == 0:
                continue
            vals = vals.iloc[nz[0]:].dropna()

            if (vals > 0).sum() < 6:
                continue

            alpha, beta = theil_sen_log_trend(vals.values)
            if alpha is None:
                continue

            for i, (c, v) in enumerate(vals.items()):
                m = month_from_label(c)
                if m in med and v > 0:
                    r = np.log(v) - (alpha + beta * i)
                    med[m].append(r)

        return {
            m: (float(np.median(v)) if v else 0.0)
            for m, v in med.items()
        }

    GLOBAL_MED = global_month_medians(df_hist_wide, time_cols, CALIBRATION_MONTHS)
    GLOBAL_CALIBRATION_FACTORS = {
        m: math.exp(GLOBAL_MED[m]) for m in CALIBRATION_MONTHS
    }

    print("Fattori globali per mesi target:",
          {m: round(GLOBAL_CALIBRATION_FACTORS[m], 4) for m in CALIBRATION_MONTHS})

    # --- per-SKU factors ---
    CALIBRATION_FACTORS = {}  # SKU -> {month -> factor}

    hist_by_sku = df_hist_wide.set_index(ID_COL)

    for sku, row_hist in hist_by_sku.iterrows():
        vals = pd.Series([row_hist[c] for c in time_cols], index=time_cols, dtype="float")
        vals = pd.to_numeric(vals, errors="coerce")

        nz = np.where((vals.fillna(0) > 0).values)[0]
        if len(nz) == 0:
            continue

        # trim solo zeri iniziali, mantieni zeri interni
        vals_trimmed = vals.iloc[nz[0]:].dropna()

        if (vals_trimmed > 0).sum() >= 6:
            alpha, beta = theil_sen_log_trend(vals_trimmed.values)
            if alpha is not None:
                bucket = {m: [] for m in CALIBRATION_MONTHS}
                for i, (c, v) in enumerate(vals_trimmed.items()):
                    m = month_from_label(c)
                    if m in bucket and v > 0:
                        r = np.log(v) - (alpha + beta * i)
                        bucket[m].append(r)

                per_month_med = {}
                for m in CALIBRATION_MONTHS:
                    if len(bucket[m]) >= 2:
                        per_month_med[m] = float(np.median(bucket[m]))
                    else:
                        per_month_med[m] = GLOBAL_MED[m]
            else:
                per_month_med = {m: GLOBAL_MED[m] for m in CALIBRATION_MONTHS}
        else:
            per_month_med = {m: GLOBAL_MED[m] for m in CALIBRATION_MONTHS}

        factors = {
            m: math.exp(per_month_med[m]) for m in CALIBRATION_MONTHS
        }
        CALIBRATION_FACTORS[sku] = factors

    print("✅ Fattori di calibrazione per-SKU calcolati (Bidirezionali).")

In [ ]:
# ============================================================
# MODULO D — CALIBRAZIONE
# D.2 — Funzione helper: get_calibration_factor_for_sku_month
# ============================================================

def get_calibration_factor_for_sku_month(sku, month):
    """
    Restituisce il fattore di calibrazione per uno SKU e un mese (1-12),
    usando:
      - fattore per-SKU se disponibile
      - altrimenti fattore globale di quel mese
      - altrimenti 1.0 (nessuna calibrazione)
    """
    if not CALIBRATION_MONTHS:
        return 1.0

    # se non rientra nei mesi da calibrare → nessun aggiustamento
    if month not in CALIBRATION_MONTHS:
        return 1.0

    # per-SKU
    if sku in CALIBRATION_FACTORS and month in CALIBRATION_FACTORS[sku]:
        return CALIBRATION_FACTORS[sku][month]

    # fallback globale
    if month in GLOBAL_CALIBRATION_FACTORS:
        return GLOBAL_CALIBRATION_FACTORS[month]

    return 1.0


In [ ]:
# ============================================================
# MODULO E — ROUNDING
# E.1 — Funzione di arrotondamento: round_to_pack
# ============================================================

def round_to_pack(value, pack, mode=ROUNDING_MODE, decimals=ROUND_DECIMALS):
    if pd.isna(value):
        return value
    if pack is None or pack <= 0:
        return round(float(value), decimals) if decimals is not None else float(value)

    scaled = float(value) / pack

    if mode == "up":
        scaled = np.ceil(scaled)
    elif mode == "down":
        scaled = np.floor(scaled)
    else:  # nearest
        scaled = np.round(scaled)

    result = scaled * pack
    return round(result, decimals) if decimals is not None else result


In [ ]:
# ============================================================
# MODULO F — MODELLO TIMESFM (Colab/Python 3.12 Safe Loader)
# ============================================================

import os, sys, importlib, pathlib, re
import numpy as np

# 1️⃣ Imposta path cache Hugging Face (velocizza i reload)
os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

MODEL_ID = "google/timesfm-2.5-200m-pytorch"

# 2️⃣ Installa dipendenze richieste dal repository (NON il pacchetto)
!pip install -q einops huggingface_hub torch

# 3️⃣ Scarica il repository ufficiale TimesFM (solo codice, non pacchetto pip)
if not pathlib.Path("/content/timesfm").exists():
    !git clone -q https://github.com/google-research/timesfm.git /content/timesfm

# 4️⃣ Aggiungi la cartella del codice a sys.path
SRC_DIR = "/content/timesfm"
PKG_DIR = "/content/timesfm/src"
T25_DIR = "/content/timesfm/src/timesfm/timesfm_2p5"

if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

# 5️⃣ Import manuale del modulo Torch correttamente
# preferiamo i file che contengono "pytorch" nel nome
torch_files = list(pathlib.Path(T25_DIR).glob("**/*pytorch*.py"))
if not torch_files:
    torch_files = list(pathlib.Path(T25_DIR).glob("**/*torch*.py"))
torch_files.sort()

if not torch_files:
    raise RuntimeError("❌ Non trovo il modulo Torch in timesfm_2p5!")

torch_mod_path = torch_files[0]
print("🧭 Modulo trovato:", torch_mod_path)

# Import del modulo
spec = importlib.util.spec_from_file_location(
    "timesfm.timesfm_2p5.timesfm_2p5_torch",
    str(torch_mod_path)
)
torch_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(torch_mod)
sys.modules["timesfm.timesfm_2p5.timesfm_2p5_torch"] = torch_mod

# 6️⃣ Trova la classe del modello
ModelClass = None
for name, cls in vars(torch_mod).items():
    if isinstance(cls, type) and hasattr(cls, "forecast"):
        ModelClass = cls
        break

if ModelClass is None:
    raise RuntimeError("❌ Nessuna classe TimesFM con metodo forecast trovata!")

print("🧩 Classe modello:", ModelClass)

# 7️⃣ Carica i pesi dal repo HuggingFace
from huggingface_hub import hf_hub_download

print("⬇️ Download pesi (HuggingFace)…")
model = ModelClass.from_pretrained(MODEL_ID)

# 8️⃣ Configurazione forecast (compatibile multipla)
cfg = None
for mp in [
    "timesfm.config",
    "timesfm.configs",
    "timesfm.timesfm_2p5.configs.forecast_config",
]:
    try:
        m = importlib.import_module(mp)
        if hasattr(m, "ForecastConfig"):
            ForecastConfig = m.ForecastConfig
            cfg = ForecastConfig(
                max_context=512,
                max_horizon=HORIZON,
                normalize_inputs=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            )
            break
    except:
        pass

if cfg is not None:
    model.compile(cfg)
else:
    model.compile()

# 9️⃣ Device (CPU/GPU)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Informazione sul device
print(f"🖥️ Device preferito: {DEVICE}")

# TimesFM non deriva da torch.nn.Module → niente .to(), .eval(), .train()
if hasattr(model, "to"):
    try:
        model.to(DEVICE)
        print("✔️ Modello spostato su device tramite .to()")
    except Exception:
        print("⚠️ .to() disponibile ma non utilizzabile — ignorato.")
else:
    print("ℹ️ Modello senza .to(): TimesFM gestisce automaticamente CPU/GPU.")

if hasattr(model, "eval"):
    try:
        model.eval()
        print("✔️ Modalità eval attivata")
    except Exception:
        print("⚠️ .eval() disponibile ma non chiamabile — ignorato.")
else:
    print("ℹ️ Modello senza .eval(): TimesFM è già in modalità inference.")

print("✔️ Modello pronto (loader manuale TimesFM attivato).")

# 🔬 Smoke test
try:
    test = np.array([10,12,11,13,15,14,16,18,17,19], dtype=np.float32)
    pred, _ = model.forecast(horizon=3, inputs=[test])
    print("🔬 Smoke test OK →", pred[0])
except Exception as e:
    print("⚠️ Smoke test fallito:", e)


In [ ]:
# ============================================================
# MODULO F — MODELLO TIMESFM
# F.2 — Funzione batch forecast_all_skus_point (Colab-safe)
# ============================================================

def forecast_all_skus_point(series_dict, horizon):
    """
    Esegue il forecast point (mediana) per tutte le serie SKU.
    Compatibile con loader manuale TimesFM su Colab (Python 3.12).
    """
    results = {}
    errors = {}

    total = len(series_dict)
    print(f"📦 Avvio forecast point per {total} SKU...\n")

    for idx, (sku, values) in enumerate(series_dict.items(), start=1):

        if VERBOSE:
            print(f"[{idx}/{total}] SKU {sku}... ", end="")

        # float32 richiesto da TimesFM
        hist = np.array(values, dtype=np.float32)

        # Device safety: spostiamo il modello su GPU/CPU
        try:
            point_fc, _ = model.forecast(
                horizon=horizon,
                inputs=[hist]
            )

            # point_fc potrebbe essere tensore Torch
            fc = point_fc[0]
            if hasattr(fc, "cpu"):
                fc = fc.cpu().numpy()

            results[sku] = fc

            if VERBOSE:
                print("OK")

        except Exception as e:
            if VERBOSE:
                print("❌ Errore")
            errors[sku] = str(e)

    print("\n✔️ Forecast point completato.")
    print(" - SKU riusciti:", len(results))
    print(" - SKU falliti:", len(errors))

    return results, errors

print("La funzione forecast_all_skus_point(...) è pronta all'uso.")


In [ ]:
# ============================================================
# MODULO G — BACKTEST (CONSISTENT THEIL-SEN & NO LEAKAGE & FINAL KPI)
# ============================================================

import numpy as np
import pandas as pd
import math

# --- 1. Calcolo Fattori Stagionali On-The-Fly (Metodo coerente col futuro) ---
def calculate_seasonality_theil_sen_local(hist_bt, dates_bt, calibration_months):
    """
    Replica la logica del Modulo D ma usando SOLO i dati del backtest.
    Usa theil_sen_log_trend (definita nel Modulo D): Theil-Sen completo,
    zeri interni preservati, posizioni temporali originali coerenti.
    """
    if not calibration_months:
        return {}

    ts = pd.Series(hist_bt, index=dates_bt)
    ts = ts[ts > 0]  # solo per il check minimo e il loop residui

    if len(ts) < 6:
        return {}

    # Stima trend sull'array completo (con zeri interni): le posizioni
    # temporali originali sono preservate internamente da theil_sen_log_trend.
    alpha, beta = theil_sen_log_trend(hist_bt)

    if alpha is None:
        return {}

    # Calcola residui detrendizzati: Residuo = Log(Reale) - (Alpha + Beta*t)
    month_residuals = {m: [] for m in calibration_months}

    for date, val in ts.items():
        m = date.month
        if m in calibration_months:
            try:
                # t = posizione nella serie completa (con zeri),
                # coerente con come theil_sen_log_trend ha stimato alpha/beta
                t = dates_bt.tolist().index(date)
                expected_log = alpha + beta * t
                actual_log = np.log(val)
                res = actual_log - expected_log
                month_residuals[m].append(res)
            except ValueError:
                continue

    factors = {}
    for m in calibration_months:
        residuals = month_residuals[m]
        if len(residuals) >= 1:
            med_res = np.median(residuals)
            factors[m] = math.exp(med_res)
        else:
            factors[m] = 1.0

    return factors

# --- 2. Funzione Backtest Single SKU ---
def backtest_single_sku_consistent(sku, hist_bt, actuals, df_filtered, model, quantile_grid):

    hist_np = np.array(hist_bt, dtype=np.float32)
    actuals = np.array(actuals, dtype=float)

    sku_data = df_filtered[df_filtered[ID_COL] == sku].sort_values("Date")
    full_dates = sku_data["Date"].tolist()

    # === FIX LENGTH MISMATCH ===
    # hist_bt è stato trimmato (zeri iniziali rimossi) nel Modulo C.
    # full_dates contiene anche le date degli zeri iniziali.
    # Allineiamo le date alla FINE dello storico (prima del test).

    end_idx = len(full_dates) - len(actuals)
    start_idx = max(0, end_idx - len(hist_bt))

    hist_bt_dates = full_dates[start_idx : end_idx]

    min_len = min(len(hist_bt), len(hist_bt_dates))
    hist_bt = hist_bt[-min_len:]
    hist_bt_dates = hist_bt_dates[-min_len:]
    hist_np = np.array(hist_bt, dtype=np.float32)

    test_months = [d.month for d in full_dates[end_idx:]]

    # --- CALIBRAZIONE (Theil-Sen No Leakage) ---
    local_factors = calculate_seasonality_theil_sen_local(
        hist_bt,
        pd.Series(hist_bt_dates),
        CALIBRATION_MONTHS
    )

    # --- Pack Size ---
    row_meta = sku_data[[PACK_SIZE_COL, UOM_COL]].iloc[0]
    pack_val = row_meta[PACK_SIZE_COL]
    pack = float(pack_val) if pd.notna(pack_val) and pack_val not in ("", None) else 1.0

    # --- TimesFM Forecast ---
    try:
        base_fc, _ = model.forecast(horizon=len(actuals), inputs=[hist_np])
        base_fc = base_fc[0]
        if hasattr(base_fc, "cpu"): base_fc = base_fc.cpu().numpy()
    except Exception:
        return None, {}, None

    accuracy_map = {}
    fc_map = {}

    # --- Grid Search ---
    for q in quantile_grid:
        scale = q / 0.5
        fc_scaled = base_fc * scale

        fc_calibrated = []
        for i, val in enumerate(fc_scaled):
            if i >= len(test_months): break
            m = test_months[i]
            factor = local_factors.get(m, 1.0)
            fc_calibrated.append(val * factor)

        fc_calibrated = np.array(fc_calibrated, dtype=float)

        fc_rounded = np.array([
            round_to_pack(v, pack, mode=ROUNDING_MODE, decimals=ROUND_DECIMALS)
            for v in fc_calibrated
        ], dtype=float)

        acc = accuracy_weighted(actuals, fc_rounded)
        accuracy_map[q] = acc
        fc_map[q] = fc_rounded

    if not accuracy_map:
        return None, {}, None

    best_q = max(accuracy_map, key=lambda x: accuracy_map[x])
    best_fc = fc_map[best_q]

    return best_q, accuracy_map, best_fc

# --- 3. ESECUZIONE BATCH ---
results_list = []
print(f"▶️ Avvio Backtest 'Theil-Sen Consistent' su {len(backtest_series)} SKU...")

for idx, sku in enumerate(backtest_series.keys(), start=1):
    hist_bt = backtest_series[sku]
    act_bt  = backtest_actuals[sku]

    if len(hist_bt) < MIN_HISTORY_POINTS or len(act_bt) < 1:
        continue

    best_q, acc_map, best_fc = backtest_single_sku_consistent(
        sku=sku,
        hist_bt=hist_bt,
        actuals=act_bt,
        df_filtered=df_filtered,
        model=model,
        quantile_grid=QUANTILE_GRID
    )

    if best_q is None:
        continue

    total_sku_weight = np.sum(act_bt + best_fc)

    results_list.append({
        "SKU": sku,
        "BestQuantile": best_q,
        "BestAccuracy": acc_map[best_q],
        "TotalWeight": total_sku_weight
    })

    if VERBOSE and idx % 10 == 0:
        print(f"[{idx}] {sku} -> BestQ: {best_q}, Acc: {acc_map[best_q]:.2f}")

df_backtest_results = pd.DataFrame(results_list)
print("\n✔️ Backtest completato.")

if not df_backtest_results.empty:
    mean_acc_simple = df_backtest_results["BestAccuracy"].mean()

    total_w = df_backtest_results["TotalWeight"].sum()
    if total_w > 0:
        weighted_acc_global = (df_backtest_results["BestAccuracy"] * df_backtest_results["TotalWeight"]).sum() / total_w
    else:
        weighted_acc_global = 0.0

    print(f"\n📊 RISULTATI BACKTEST:")
    print(f" - SKU Processati: {len(df_backtest_results)}")
    print(f" - Media Semplice (SKU equal weight):  {mean_acc_simple:.2%}")
    print(f" - MEDIA PESATA GLOBALE (Volume weight): {weighted_acc_global:.2%}  <-- IL TUO KPI REALE")

    display(df_backtest_results.head())
else:
    print("⚠️ Nessun risultato generato.")

In [ ]:
# ============================================================
# MODULO H — FORECAST FUTURO
# H.1 — Forecast point TimesFM (Colab-safe)
# ============================================================

print("➡️ Avvio forecast (TimesFM)")
print(f"Horizon scelto: {HORIZON} mesi\n")

fc_results, fc_errors = forecast_all_skus_point(
    series_dict=sku_series,
    horizon=HORIZON
)

print("\n📊 Riepilogo forecast:")
print(" - SKU con forecast:", len(fc_results))
print(" - SKU con errori:", len(fc_errors))

if fc_errors:
    print("\n❌ Errori rilevati su:")
    for sku, err in fc_errors.items():
        if VERBOSE:
            print(f"   SKU {sku}: {err}")
else:
    print("✔️ Nessun errore nei forecast.")


In [ ]:
# ============================================================
# MODULO H — FORECAST FUTURO
# H.2 — Forecast futuro (quantile ottimale + calibrazione + rounding)
# ============================================================

fc_rows = []

print("➡️ Ricostruzione forecast per-SKU su timeline globale (pipeline completa)...")

# Timeline futura globale
last_date_global = df_filtered["Date"].max()
print("Ultima data storica globale:", last_date_global)

start_date_global = last_date_global + pd.offsets.MonthBegin(1)
future_dates_global = [
    start_date_global + pd.offsets.MonthBegin(i)
    for i in range(HORIZON)
]

# 🔹 Mappa SKU -> best quantile (dal backtest)
best_q_map = {}
if "df_backtest_results" in globals():
    best_q_map = dict(zip(df_backtest_results["SKU"], df_backtest_results["BestQuantile"]))
else:
    print("⚠️ Attenzione: df_backtest_results non trovato, uso q=0.5 per tutti gli SKU.")

def get_best_quantile_for_sku(sku):
    """
    Restituisce il best quantile per lo SKU, o 0.5 se non disponibile.
    """
    q = best_q_map.get(sku, 0.5)
    try:
        return float(q)
    except Exception:
        return 0.5

# 🔹 Mappa SKU -> pack-size
meta_pack = (
    df_filtered[[ID_COL, PACK_SIZE_COL, UOM_COL]]
    .drop_duplicates()
    .set_index(ID_COL)
)

def get_pack_for_sku(sku):
    """
    Restituisce il pack-size per lo SKU, default = 1.0 se mancante o non numerico.
    """
    if sku not in meta_pack.index:
        return 1.0
    v = meta_pack.loc[sku, PACK_SIZE_COL]
    if pd.isna(v) or v in ("", None):
        return 1.0
    try:
        return float(v)
    except Exception:
        return 1.0

# 🔁 Loop su tutti gli SKU con forecast point
for sku, base_fc in fc_results.items():
    base_fc = np.array(base_fc, dtype=float)

    # Best quantile per questo SKU (o 0.5)
    q = get_best_quantile_for_sku(sku)
    scale = q / 0.5  # scaling rispetto alla mediana

    # Forecast scalato per quantile
    fc_scaled = base_fc * scale

    # Pack-size per rounding
    pack = get_pack_for_sku(sku)

    # Allineiamo lunghezza forecast e date future
    steps = min(len(fc_scaled), len(future_dates_global))

    for i in range(steps):
        date = future_dates_global[i]
        month = int(date.month)

        # Calibrazione per mese (stessa logica del backtest)
        factor = get_calibration_factor_for_sku_month(sku, month)
        fc_calibrated = fc_scaled[i] * factor

        # Rounding secondo pack-size
        fc_final = round_to_pack(
            value=fc_calibrated,
            pack=pack,
            mode=ROUNDING_MODE,
            decimals=ROUND_DECIMALS
        )

        fc_rows.append({
            ID_COL: sku,
            "Date": date,
            "Period": date.strftime("%Y_%m"),
            "Forecast": float(fc_final),      # già scalato + calibrato + arrotondato
            "BestQuantile": q                 # per tracciabilità
        })

df_fc_long = pd.DataFrame(fc_rows)

print("Preview forecast LONG (già scalato + calibrato + arrotondato):")
df_fc_long.head()


In [ ]:
# ============================================================
# MODULO H — FORECAST FUTURO
# H.3 — Conversione forecast LONG → WIDE
# ============================================================

print("➡️ Conversione forecast LONG → WIDE (valori finali)...")

df_fc_wide = df_fc_long.pivot(
    index=ID_COL,
    columns="Period",
    values="Forecast"   # valori già finali
).reset_index()

# ordina SOLO le colonne forecast
forecast_cols = sorted([c for c in df_fc_wide.columns if re.match(r"^\d{4}_\d{2}$", str(c))])
ordered_cols = [ID_COL] + forecast_cols
df_fc_wide = df_fc_wide[ordered_cols]

print("Preview forecast WIDE (finale):")
df_fc_wide.head()


In [ ]:
# ============================================================
# MODULO I — OUTPUT FINALE
# I.1 — Merge: meta + storico + forecast finale
# ============================================================

print("➡️ Costruzione tabella finale (storico + forecast finale)...")

# 1️⃣ Metadati SKU
meta = df_filtered[[ID_COL, DESC_COL, PACK_SIZE_COL, UOM_COL]].drop_duplicates()

# 2️⃣ Storico WIDE (domanda reale)
df_hist_wide = df_filtered.pivot(
    index=ID_COL,
    columns="Period",
    values="Demand"
).reset_index()

# 3️⃣ Rinomina colonne forecast WIDE con prefisso "f"
df_fc_pref = df_fc_wide.copy()
df_fc_pref.rename(
    columns={c: f"f{c}" for c in df_fc_pref.columns if c != ID_COL},
    inplace=True
)

# 4️⃣ Merge: meta + storico + forecast
out_final = meta.merge(df_hist_wide, on=ID_COL, how="left")
out_final = out_final.merge(df_fc_pref, on=ID_COL, how="left")

print("✔️ Tabella finale generata (storico + forecast scalato, calibrato e arrotondato).")
out_final.head()


In [ ]:
# ============================================================
# MODULO J — INVENTORY PLANNING (ABC, XYZ, SS) - Parametrico
# ============================================================
from scipy.stats import norm

def calculate_inventory_logic(df_history, meta_df):
    print("➡️ Avvio calcolo ABC / XYZ / Safety Stock...")

    # 1. Preparazione Dati (Ultimi N mesi di storico WINSORIZZATO)
    last_date = df_history["Date"].max()
    start_date_ss = last_date - pd.DateOffset(months=SS_LOOKBACK_MONTHS-1)

    df_ss = df_history[df_history["Date"] >= start_date_ss].copy()

    # Raggruppa per SKU
    stats = df_ss.groupby(ID_COL)["Demand"].agg(["sum", "mean", "std"]).reset_index()
    stats.rename(columns={"sum": "TotalVol", "mean": "AvgDemand", "std": "StdDev"}, inplace=True)
    stats["StdDev"] = stats["StdDev"].fillna(0)

    # 2. Calcolo ABC (Pareto sui Volumi)
    stats = stats.sort_values("TotalVol", ascending=False)
    total_volume_global = stats["TotalVol"].sum()

    if total_volume_global <= 0:
        # Nessun volume nel lookback: tutti in classe C
        stats["ABC"] = "C"
    else:
        stats["CumVol"] = stats["TotalVol"].cumsum()
        stats["CumPerc"] = stats["CumVol"] / total_volume_global

        def get_abc(p):
            if p <= ABC_LIMITS["A"]: return "A"
            elif p <= ABC_LIMITS["B"]: return "B"
            return "C"

        stats["ABC"] = stats["CumPerc"].apply(get_abc)

    # 3. Calcolo XYZ (Volatilità)
    stats["CV"] = stats["StdDev"] / stats["AvgDemand"]
    stats["CV"] = stats["CV"].replace([np.inf, -np.inf], 999.0).fillna(0)

    def get_xyz(cv):
        if cv <= XYZ_LIMITS["X"]: return "X"
        elif cv <= XYZ_LIMITS["Y"]: return "Y"
        return "Z"

    stats["XYZ"] = stats["CV"].apply(get_xyz)

    # 4. Combo Class
    stats["Class"] = stats["ABC"] + stats["XYZ"]

    # 5. Recupero Lead Time (LT) e Pack Size (Round)
    if LT_COL_NAME in meta_df.columns:
        lt_map = meta_df[[ID_COL, LT_COL_NAME]].drop_duplicates(subset=ID_COL)
        stats = stats.merge(lt_map, on=ID_COL, how="left")
        stats["LT_Final"] = stats[LT_COL_NAME].fillna(DEFAULT_LEAD_TIME)
    else:
        stats["LT_Final"] = DEFAULT_LEAD_TIME

    if PACK_SIZE_COL in meta_df.columns:
        pack_map = meta_df[[ID_COL, PACK_SIZE_COL]].drop_duplicates(subset=ID_COL)
        stats = stats.merge(pack_map, on=ID_COL, how="left")
    else:
        stats[PACK_SIZE_COL] = 1.0

    # 6. Calcolo Safety Stock
    ss_values = []

    for _, row in stats.iterrows():
        cls = row["Class"]
        sigma = row["StdDev"]
        lt = row["LT_Final"]

        pack_val = row.get(PACK_SIZE_COL, 1.0)
        try:
            pack = float(pack_val) if pd.notna(pack_val) and pack_val > 0 else 1.0
        except:
            pack = 1.0

        sl_target = SERVICE_LEVEL_MATRIX.get(cls, 0.0)

        if sl_target <= 0 or sigma <= 0:
            ss_values.append(0.0)
            continue

        z_score = norm.ppf(sl_target)
        time_factor = np.sqrt((lt + REORDER_PERIOD) / 30.0)

        ss_raw = z_score * sigma * time_factor
        ss_final = round_to_pack(ss_raw, pack, mode="up", decimals=ROUND_DECIMALS)

        ss_values.append(ss_final)

    stats["SafetyStock"] = ss_values

    cols_to_keep = [ID_COL, "LT_Final", "ABC", "XYZ", "SafetyStock"]
    return stats[cols_to_keep]

if CALCULATE_SS:
    df_inventory = calculate_inventory_logic(df_filtered, df_raw)
    print("\nEsempio Inventory Logic (SS arrotondato UP all'imballo con decimali corretti):")
    display(df_inventory.head())
else:
    df_inventory = None

In [ ]:
# ============================================================
# MODULO K — OUTPUT FINALE
# ============================================================

print("➡️ Costruzione tabella finale (Meta + Inventory + Storico + Forecast)...")

# 1. Metadati Base
meta = df_filtered[[ID_COL, DESC_COL, PACK_SIZE_COL, UOM_COL]].drop_duplicates()

# 2. Inventory Data (Merge)
if df_inventory is not None:
    # Rinomina LT_Final in LT per pulizia
    df_inv_clean = df_inventory.rename(columns={"LT_Final": "LT"})
    out_step1 = meta.merge(df_inv_clean, on=ID_COL, how="left")
else:
    out_step1 = meta.copy()

# 3. Storico WIDE
df_hist_wide = df_filtered.pivot(
    index=ID_COL,
    columns="Period",
    values="Demand"
).reset_index()

# 4. Forecast WIDE (con prefisso "f")
df_fc_pref = df_fc_wide.copy()
df_fc_pref.rename(
    columns={c: f"f{c}" for c in df_fc_pref.columns if c != ID_COL},
    inplace=True
)

# 5. Merge Finale Sequenziale per ordinare le colonne
out_final = out_step1.merge(df_hist_wide, on=ID_COL, how="left")
out_final = out_final.merge(df_fc_pref, on=ID_COL, how="left")

# Forza SKU stringa
out_final[ID_COL] = out_final[ID_COL].astype(str)

# Gestione NaN su colonne inventory (per sicurezza)
if "SafetyStock" in out_final.columns:
    out_final["SafetyStock"] = out_final["SafetyStock"].fillna(0)

print(f"✔️ Tabella finale pronta. Shape: {out_final.shape}")

# Export
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_BASE + OUTPUT_SUFFIX + ".xlsx")
out_final.to_excel(output_path, index=False)
print(f"✔️ File salvato in {output_path}")
files.download(output_path)